In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np

load_dotenv()
client = OpenAI()

In [2]:
def find_most_similar(query: str, corpus: list[str]) -> list[tuple[float, str]]:
    all_texts = [query] + corpus
    all_embeddings = []

    response = client.embeddings.create(
        input=all_texts,
        model="text-embedding-3-small",
    )

    all_embeddings = [item.embedding for item in response.data]

    query_emb = np.array(all_embeddings[0])
    corpus_embs = np.array(all_embeddings[1:])

    # Vectorised cosine similarity — much faster than loop
    # Normalise all vectors, then dot product = cosine similarity
    query_norm = query_emb / np.linalg.norm(query_emb)
    corpus_norms = corpus_embs / np.linalg.norm(corpus_embs, axis=1, keepdims=True)
    similarities = corpus_norms @ query_norm

    ranked = sorted(zip(similarities, corpus), key=lambda x: x[0], reverse=True)
    return ranked



In [3]:
corpus = [
    "PostgreSQL indexes improve query performance",
    "Redis caching reduces database load",
    "JWT tokens expire after 60 minutes",
    "Docker containers share the host kernel",
    "B-tree indexes support range queries",
]

results = find_most_similar("how to speed up database queries", corpus)
for score, text in results:
    print(f"{score:.4f} - {text}")

0.5541 - PostgreSQL indexes improve query performance
0.5166 - Redis caching reduces database load
0.4218 - B-tree indexes support range queries
0.1712 - JWT tokens expire after 60 minutes
0.0543 - Docker containers share the host kernel
